In [ ]:
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# ============================================
# LOAD AUDIO
# ============================================

audio, sr = librosa.load("audio.mp3", sr=None, mono=True)

audio = audio.astype(np.float32)
audio /= np.max(np.abs(audio))

clean = audio.copy()

# ============================================
# SPLIT NOT BY TIME (same signal reused)
# ============================================

split_idx = int(0.7 * len(clean))

train_clean = clean[:split_idx]
test_clean  = clean[:split_idx]   # SAME REGION (important for Case 3)

# ============================================
# DIFFERENT NOISE FOR TRAIN AND TEST
# ============================================

noise_strength = 0.02

train_noisy = train_clean + noise_strength * np.random.randn(len(train_clean))
test_noisy  = test_clean  + noise_strength * np.random.randn(len(test_clean))

train_noisy = np.clip(train_noisy, -1, 1)
test_noisy  = np.clip(test_noisy, -1, 1)

# ============================================
# CREATE DATASET (TRAIN ONLY)
# ============================================

window_size = 1024
hop_size = 512

X_train, Y_train = [], []

for i in range(0, len(train_clean) - window_size, hop_size):

    x = train_noisy[i:i+window_size]
    y = train_clean[i:i+window_size]

    X_train.append(x)
    Y_train.append(y)

X_train = np.array(X_train)
Y_train = np.array(Y_train)

print("Train samples:", X_train.shape)

# ============================================
# MODEL: 3-TAP FIR FILTER
# ============================================

a = np.random.randn(3).astype(np.float32)

lr = 1e-6
epochs = 50

loss_history = []

# ============================================
# TRAINING (GRADIENT DESCENT)
# ============================================

for epoch in range(epochs):

    total_loss = 0.0
    grad = np.zeros(3)

    for x, y in zip(X_train, Y_train):

        pred = (
            a[0] * x[1:-1] +
            a[1] * x[:-2] +
            a[2] * x[2:]
        )

        target = y[1:-1]

        error = pred - target

        total_loss += np.mean(error**2)

        grad[0] += np.sum(2 * error * x[1:-1])
        grad[1] += np.sum(2 * error * x[:-2])
        grad[2] += np.sum(2 * error * x[2:])

    a -= lr * grad
    loss_history.append(total_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch} | Loss: {total_loss:.6f}")

print("\nLearned filter:", a)

# ============================================
# TEST PHASE (DIFFERENT NOISE INSTANCE)
# ============================================

denoised_test = np.zeros_like(test_clean)

denoised_test[1:-1] = (
    a[0] * test_noisy[1:-1] +
    a[1] * test_noisy[:-2] +
    a[2] * test_noisy[2:]
)

# ============================================
# SAVE FILES
# ============================================

sf.write("train_noisy.wav", train_noisy, sr)
sf.write("test_noisy.wav", test_noisy, sr)
sf.write("test_denoised.wav", denoised_test, sr)

print("\nSaved files.")

# ============================================
# PLAY RESULTS
# ============================================

print("\nTEST CLEAN")
display(Audio(test_clean, rate=sr))

print("\nTEST NOISY")
display(Audio(test_noisy, rate=sr))

print("\nTEST DENOISED")
display(Audio(denoised_test, rate=sr))

# ============================================
# SNR EVALUATION
# ============================================

signal_power = np.mean(test_clean ** 2)

noise_before = np.mean((test_clean - test_noisy) ** 2)
noise_after  = np.mean((test_clean - denoised_test) ** 2)

snr_before = 10 * np.log10(signal_power / noise_before)
snr_after  = 10 * np.log10(signal_power / noise_after)

print("\n===== SNR RESULTS (CASE 3) =====")
print(f"Before: {snr_before:.2f} dB")
print(f"After : {snr_after:.2f} dB")

# ============================================
# LOSS PLOT
# ============================================

plt.plot(loss_history)
plt.title("Training Loss (Case 3)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()